In [ ]:
!pip install -U bitsandbytes

import numpy as np
from tqdm.auto import tqdm

from datasets import load_dataset

import plotly.graph_objects as go
from plotly.subplots import make_subplots

dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

print(f"Number of samples in the training set: {len(dataset['train'])}")
print('-'*20)

prob = 1

print(dataset['train'][prob]['input'])
print('-'*20)
print(dataset['train'][prob]['output'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.5 MB/s eta 0:00:00


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…): reconstructing file:   0%|          |  0.00B / 70.5MB            

data/train-00000-of-00001-5e7cb295b9cff0(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Number of samples in the training set: 112165
--------------------
My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!
--------------------
Hi... Thank you for consulting in Chat Doctor. It seems your kid is having viral diarrhea. Once it starts it will take 5-7 days to completely get better. Unless the kids having low urine output or very dull or excessively sleepy or blood in motion or green bilious vomiting...you need not worry. There is no need to use antibiotics unless there is blood in the motion. Antibiotics might worsen if unnecessarily used causing antibiotic associated diarrhea. I suggest you use zinc supplements (Z&D Chat Doctor.


In [ ]:
input_word_counts = [len(example['input'].split()) for example in tqdm(dataset['train'])]
output_word_counts = [len(example['output'].split()) for example in tqdm(dataset['train'])]

average_input_words = np.mean(input_word_counts)
average_output_words = np.mean(output_word_counts)

print(average_input_words)
print(average_output_words)

  0%|          | 0/112165 [00:00<?, ?it/s]

  0%|          | 0/112165 [00:00<?, ?it/s]

81.13825168278875
102.1110506842598


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Distribution of Input Word Counts', 'Distribution of Output Word Counts'))

fig.add_trace(go.Histogram(x=input_word_counts, nbinsx=50), row=1, col=1)
fig.add_trace(go.Histogram(x=output_word_counts, nbinsx=50), row=1, col=2)

fig.update_layout(
    height=600,
    width=1200,
    title=dict(
        text="<b>Distribution of Word Counts</b>",
        x=0.5,
        xanchor='center',
        font=dict(
            family="monospace",
            size=22, # You can adjust the size if needed
            color="black" # You can adjust the color if needed
        )
    ),
    font=dict(
        family="monospace"
    )
)
fig.update_xaxes(title_text="Number of Words", row=1, col=1)
fig.update_yaxes(title_text="Frequency", row=1, col=1)
fig.update_xaxes(title_text="Number of Words", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.show()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

### Load Model with 4-bit Quantization using Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id, # Your pre-trained model ID
    max_seq_length = 512, # The maximum sequence length for the model
    dtype = None, # Autodetect based on accelerator
    load_in_4bit = True, # Load model in 4-bit precision
    # token = "hf_...
)

# Configure LoRA for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA attention dimension
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16, # Alpha parameter for LoRA scaling
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none", # Do not add bias to LoRA layers
    use_gradient_checkpointing = True, # Enable gradient checkpointing for memory efficiency
    random_state = 3407, # Random seed for reproducibility
    use_rslora = False, # Use regular LoRA
    loftq_config = None, # No LoftQ configuration
)

display(model)
display(tokenizer)

In [ ]:
def tokenize_function(examples):

    text = [inp + " " + out for inp, out in zip(examples["input"], examples["output"])]
    return tokenizer(text, truncation=True, padding="max_length", max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])

display(tokenized_datasets['train'])

Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input_ids', 'attention_mask'],
    num_rows: 112165
})

In [ ]:
dataset['train'][0]['input']

'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!'

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results",          # Output directory for checkpoints and predictions
    per_device_train_batch_size=4,   # Batch size for training
    gradient_accumulation_steps=2,   # Accumulate gradients over 2 batches
    learning_rate=2e-5,              # Learning rate
    num_train_epochs=3,              # Number of training epochs
    logging_steps=10,                # Log every 10 steps
    save_steps=500,                  # Save checkpoint every 500 steps
    save_total_limit=2,              # Limit the number of saved checkpoints
    push_to_hub=False,               # Do not push to the Hugging Face Hub
    report_to="none",                # Disable reporting to any service
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.





### 1. LoRA + Quantization

In [ ]:
2B Param - 2.5Gb
10B Param - 12 GB -> 3gb


param = [1.4852378492738957, ..., .495870324780237483] 1 Billion
param = [1.48523788, ..., .49587032] 1 Billion
param = [1.48, ..., .49] 1 Billion

32bit precision - 16GB
16 bit precision - 8GB
8 bit precision  - 4GB
4 bit precision  - 2GB
2 bit precision  - 1GB

In [ ]:
2048*5000

10240000

In [ ]:
num_rows = 2048
num_cols = 5000

total_elements = num_rows * num_cols

# A float32 (fp32) occupies 4 bytes
bytes_per_element = 4

total_bytes = total_elements * bytes_per_element
total_mb = total_bytes / (1024 * 1024)
total_gb = total_bytes / (1024 * 1024 * 1024)

print(f"Matrix shape: ({num_rows}, {num_cols})")
print(f"Total elements: {total_elements}")
print(f"Size per element (fp32): {bytes_per_element} bytes")
print(f"Total memory: {total_bytes} bytes")
print(f"Total memory: {total_mb:.2f} MB")
print(f"Total memory: {total_gb:.2f} GB")

Matrix shape: (2048, 5000)
Total elements: 10240000
Size per element (fp32): 4 bytes
Total memory: 40960000 bytes
Total memory: 39.06 MB
Total memory: 0.04 GB
